# AD Variant Analysis – Example Notebook

This notebook walks through analyzing de novo single nucleotide variants (SNVs) in **human** transcription factor (TF) protein domains using the `ad-variant-analysis` package.

> **Note:** The CDS BED files (`cds_beds/`), protein sequences (`raw_files/proteins.dat`), and DNA transcript sequences (`raw_files/dna_transcripts.dat`) provided in this repository are specific to **human TFs**.

## Installation

Install the package from the repository root:

```bash
pip install .
```

The package also requires **bedtools** for genomic interval operations:

```bash
# Ubuntu / Debian
sudo apt-get install bedtools

# macOS (Homebrew)
brew install bedtools

# Conda
conda install -c bioconda bedtools
```

In [24]:
import pandas as pd
from pathlib import Path

# Package-level import to confirm installation
import AD_variant_analysis
print(f"ad-variant-analysis version: {AD_variant_analysis.__version__}")

ad-variant-analysis version: 0.1.0


## 1. Input data

### 1a. TF domain annotation table

A tab-separated file with one row per transcript, containing:
- UniProt and Ensembl IDs
- Amino acid coordinate ranges for each domain type (DBD, AD, RD, Bif, IDR)

Coordinates are 1-based and formatted as comma-separated `start-end` ranges, e.g. `10-50,60-90`.

In [25]:
all_TFs = pd.read_csv("all_TFs_table_proteins_with_IDR.txt", sep="\t", index_col=0)
print(f"{len(all_TFs):,} transcription factors loaded")
all_TFs.head()

1,590 transcription factors loaded


,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length,IDR_coords
0,NaN,NaN,A0A087WUV0,NaN,ENST00000425953,"221-243,249-271,277-299,305-327,333-355,361-38...",NaN,NaN,NaN,1,"1-216,498-523"
1,NaN,NaN,A0AVK6,NaN,ENST00000250024,"114-182,262-347",NaN,NaN,NaN,1,"1-115,337-868"
2,NaN,NaN,A1YPR0,NaN,ENST00000535628,"364-386,392-414,420-442,448-469",NaN,NaN,NaN,1,"128-364,471-620"
3,NaN,NaN,A2RRD8,NaN,ENST00000391781,"161-183,189-211,217-239,245-267,273-295,301-32...",NaN,NaN,NaN,1,51-156
4,NaN,NaN,A2RU54,NaN,ENST00000339992,150-206,NaN,NaN,NaN,1,"1-158,224-274"


In [26]:
# Summary: how many TFs have each domain annotated?
domain_cols = ["DBD_coords", "AD_coords", "RD_coords", "Bif_coords", "IDR_coords"]
domain_counts = all_TFs[domain_cols].notna().sum()
domain_counts.index = [c.replace("_coords", "") for c in domain_cols]
domain_counts.rename("TFs with annotation").to_frame()

,TFs with annotation
DBD,1590
AD,446
RD,0
Bif,0
IDR,1590


### 1b. Variants BED file

A 6-column BED file (no header) with de novo SNVs:

| col | description |
|-----|-------------|
| 0   | chromosome  |
| 1   | start (0-based) |
| 2   | end         |
| 3   | reference allele |
| 4   | alternate allele |
| 5   | score / family ID |

In [27]:
variants = pd.read_csv(
    "SFARI_TF_de_novo_variants_gpf.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "ref", "alt", "score"],
)
print(f"{len(variants):,} variants loaded")
variants.head()

117 variants loaded


,chrom,start,end,ref,alt,score
0,1,61352563,61352564,C,T,0
1,2,1917306,1917307,A,G,0
2,2,156329363,156329364,A,G,0
3,1,61352563,61352564,C,T,0
4,4,105234730,105234731,G,T,0


## 2. Pipeline

The full analysis runs as a series of CLI commands (or equivalent Python API calls).

```
CDS BED files  +  Variants BED
        |                |
        v                v
  intersect-variants          <- step 1: find variants in coding regions
        |
        v
  classify-snvs               <- step 2: Syn / Non-Syn / Nonsense

TF domain annotation + CDS BED files
        |
        v
    map-domains                <- step 3: map protein domain coords to genomic BED files
        |
        v
  (classified SNVs) + (domain BEDs)
        |
        v
  classify-domain-snvs        <- step 4: per-domain results
```

### Prerequisites

Steps 1-4 require **CDS BED files** (one per Ensembl transcript). Human TF CDS BED files are provided in `cds_beds/`.
Steps 2-4 additionally require **protein** and **DNA transcript** reference
sequences stored as Python pickle files (dict of `ENST_ID -> sequence`). These are provided for human TFs in `raw_files/proteins.dat` and `raw_files/dna_transcripts.dat`.

Set the paths below to match your local data:


In [28]:
# --- adjust these paths to your data ---
CDS_DIR             = Path("cds_beds/")                        # human TF CDS BED files (one per ENST); provided in repo
VARIANTS_BED        = Path("SFARI_TF_de_novo_variants_gpf.bed")
PROTEINS_PKL        = Path("raw_files/proteins.dat")            # human TF protein sequences (pickle)
DNA_TRANSCRIPTS_PKL = Path("raw_files/dna_transcripts.dat")     # human TF DNA transcript sequences (pickle)
TF_TABLE            = Path("all_TFs_table_proteins_with_IDR.txt")
DOMAIN_BEDS_DIR     = Path("domain_beds/")                     # genomic domain BED files (generated by map-domains)
OUTPUT_DIR          = Path("output/")

### Step 1 – Intersect variants with CDS regions

Finds which variants overlap coding sequences. Sorted files are cached in
`output/sorted/` and reused on subsequent runs.

In [29]:
! intersect-variants \
    --cds-dir    {CDS_DIR} \
    --variants   {VARIANTS_BED} \
    --output-dir {OUTPUT_DIR}

Will process all 1559 CDS files from cds_beds

Output structure:
  Sorted CDS files: output/sorted/cds
  Sorted variants:  output/sorted/variants
  Intersections:    output/intersections

Step 1: Sorting CDS files...
Sorting CDS files: 100%|█████████████████| 1559/1559 [00:00<00:00, 75875.14it/s]
  Sorted 1559/1559 CDS files

Step 2: Sorting variants file...
  Variants already sorted (using cached version)

Step 3: Intersecting CDS files with variants...
  Using sorted CDS files from: output/sorted/cds
  Using sorted variants from: output/sorted/variants/SFARI_TF_de_novo_variants_gpf.bed
  Writing intersections to: output/intersections

Processing variants: 100%|█████████████████| 1559/1559 [00:01<00:00, 804.28it/s]

COMPLETE!

Results:
  Total intersection files: 1559
  Files with variants:      36
  Files without variants:   1523

Output locations:
  Intersections: output/intersections
  Sorted files:  output/sorted (reusable for future runs)

Example files with variants:
  ENST00000

### Step 2 – Classify CDS variants

Labels each CDS variant as:
- **Syn** – synonymous (no amino acid change)
- **Non-Syn** – non-synonymous (amino acid change)
- **Nonsense** – introduces a stop codon

In [30]:
! classify-snvs \
    --input-dir       {OUTPUT_DIR}/intersections \
    --output-dir      {OUTPUT_DIR}/classified \
    --proteins        {PROTEINS_PKL} \
    --dna-transcripts {DNA_TRANSCRIPTS_PKL} \
    --sorted-cds-dir  {OUTPUT_DIR}/sorted/cds

Loading reference sequences...
  Loaded 1558 protein sequences
  Loaded 1558 DNA transcripts

Processing 1559 variant files...
  Input directory: output/intersections
  Output directory: output/classified

Processing SNVs: 100%|████████████████████| 1559/1559 [00:00<00:00, 9808.92it/s]

COMPLETE!

Results:
  Total variants processed: 115
  Successfully classified: 115
  Classification rate: 100.0%

Classification summary:
  Synonymous: 0 (0.0%)
  Non-synonymous: 115 (100.0%)
  Nonsense: 0 (0.0%)

Output location: output/classified


### Step 3 – Generate domain BED files

Maps protein domain coordinates (amino-acid space) from the TF annotation table
to genomic BED format using the CDS BED files. One output file is written per TF
(named by UniProt ID) and contains rows for every annotated domain type
(DBD, AD, RD, Bif, IDR).


In [31]:
! map-domains \
    --input            {TF_TABLE} \
    --cds-directory    {CDS_DIR} \
    --output-directory {DOMAIN_BEDS_DIR}


Processing transcripts...
  Input file: all_TFs_table_proteins_with_IDR.txt
  CDS directory: cds_beds
  Output directory: domain_beds

No CDS for ENST00000383196
No CDS for ENST00000610440
Trying to add AA 1209 but protein has length 1208
Trying to add AA 1210 but protein has length 1208
Trying to add AA 1211 but protein has length 1208
Trying to add AA 1212 but protein has length 1208
Trying to add AA 1213 but protein has length 1208
Trying to add AA 1214 but protein has length 1208
Trying to add AA 1215 but protein has length 1208
Trying to add AA 1216 but protein has length 1208
Trying to add AA 1217 but protein has length 1208
Trying to add AA 1218 but protein has length 1208
Trying to add AA 1219 but protein has length 1208
Trying to add AA 1225 but protein has length 1208
Trying to add AA 1226 but protein has length 1208
Trying to add AA 1227 but protein has length 1208
Trying to add AA 1228 but protein has length 1208
Trying to add AA 1229 but protein has length 1208
Trying to a

### Step 4 – Classify variants by protein domain

Intersects classified SNVs with each TF's domain regions (DBD, AD, RD, Bif, IDR)
to identify which variants fall within functional domains and their effect.


In [32]:
! classify-domain-snvs \
    --domain-dir         {DOMAIN_BEDS_DIR} \
    --classified-snv-dir {OUTPUT_DIR}/classified \
    --output-dir         {OUTPUT_DIR}/domain_snvs \
    --mapping            {TF_TABLE}

Loading UniProt to ENST mapping from all_TFs_table_proteins_with_IDR.txt
  Loaded 1590 protein mappings

Intersecting domain files with classified SNVs...
  Domain directory: domain_beds
  Classified SNV directory: output/classified
  Output directory: output/domain_snvs

Found 1558 domain files in domain_beds
Processing domain SNVs: 100%|██████████████| 1558/1558 [00:02<00:00, 717.54it/s]

COMPLETE!

Results:
  Successfully processed: 1558
  Total SNVs in domains: 129
  Errors/warnings: 0

Output:
  Total files: 1558
  Files with domain SNVs: 33
  Files without SNVs: 1525

SNV Classification in domains:
  Synonymous: 0 (0.0%)
  Non-synonymous: 129 (100.0%)
  Nonsense: 0 (0.0%)

Example files with domain SNVs:
  ENST00000356073.bed: 3 SNVs
  ENST00000379044.bed: 2 SNVs
  ENST00000246672.bed: 2 SNVs
  ENST00000288319.bed: 1 SNVs
  ENST00000317216.bed: 3 SNVs
  ... and 28 more

Output location: output/domain_snvs


## 3. Exploring results

After the pipeline completes, the per-domain output files can be loaded into
pandas for downstream analysis.

In [42]:
result_files = list((OUTPUT_DIR / "domain_snvs").glob("*.bed"))
non_empty = [f for f in result_files if f.stat().st_size > 0]
print(f"{len(result_files)} total output files, {len(non_empty)} with domain variants")

# Load and concatenate all non-empty results
col_names = [
    "dom_chrom", "dom_start", "dom_end", "domain_type", "ensg", "score", "strand", "enst",
    "var_chrom", "var_start", "var_end", "enst2", "strand2", "chr2", "mut_start", "mut_end", "ref_nt", "alt_nt", "score",
    "wt_aa", "mt_aa", "classification", 
]

if non_empty:
    dfs = []
    for f in non_empty:
        df = pd.read_csv(f, sep="\t", header=None)
        df.columns = col_names[: len(df.columns)]
        dfs.append(df)
    results = pd.concat(dfs, ignore_index=True)
    print(f"{len(results):,} domain variants total")
    results.head()

results

1558 total output files, 33 with domain variants
129 domain variants total


,dom_chrom,dom_start,dom_end,domain_type,ensg,score,strand,enst,var_chrom,var_start,...,strand2,chr2,mut_start,mut_end,ref_nt,alt_nt,score,wt_aa,mt_aa,classification
0,18,55228884,55228885,DBD,NaN,.,-1,ENST00000356073,18,55228884,...,-1,18,55228884,55228885,G,A,0,A,V,Non-Syn
1,18,55228986,55228987,DBD,NaN,.,-1,ENST00000356073,18,55228986,...,-1,18,55228986,55228987,C,T,0,R,Q,Non-Syn
2,18,55228993,55228994,DBD,NaN,.,-1,ENST00000356073,18,55228993,...,-1,18,55228993,55228994,G,A,0,R,C,Non-Syn
3,X,25013159,25013160,IDR,NaN,.,-1,ENST00000379044,X,25013159,...,-1,X,25013159,25013160,C,T,0,A,T,Non-Syn
4,X,25013159,25013160,IDR,NaN,.,-1,ENST00000379044,X,25013159,...,-1,X,25013159,25013160,C,T,0,A,T,Non-Syn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,9,37033985,37033986,DBD,NaN,.,-1,ENST00000358127,9,37033985,...,-1,9,37033985,37033986,C,G,0,G,R,Non-Syn
125,9,37033985,37033986,IDR,NaN,.,-1,ENST00000358127,9,37033985,...,-1,9,37033985,37033986,C,G,0,G,R,Non-Syn
126,14,64282801,64282802,AD,NaN,.,-1,ENST00000341099,14,64282801,...,-1,14,64282801,64282802,C,T,0,V,I,Non-Syn
127,14,64235128,64235129,AD,NaN,.,-1,ENST00000341099,14,64235128,...,-1,14,64235128,64235129,G,A,0,A,V,Non-Syn


In [ ]:
if non_empty:
    # Count by domain type and classification
    summary = (
        results.groupby(["domain_type", "classification"])
        .size()
        .unstack(fill_value=0)
        .rename_axis(None, axis=1)
    )
    print("Variant counts by domain and classification:")
summary

Variant counts by domain and classification:


,Non-Syn
domain_type,
AD,30
DBD,34
IDR,65
